# Búsqueda de Grover

Este notebook acompaña el material académico integrado sobre **Grover's Search**. El objetivo es estudiar el algoritmo como una técnica de búsqueda no estructurada basada en cuatro ideas: preparación de una superposición uniforme, marcado de soluciones por fase, difusión como inversión respecto a la media y amplificación de amplitud.

El notebook es autoexplicativo. Las secciones matemáticas siguen el mismo orden temático de la presentación integrada y desarrollan los ejemplos y ejercicios con pasos intermedios suficientes para verificar formalmente cada conclusión.

## Convención de notación de Dirac

Para evitar que la notación se rompa entre JupyterLab, Anaconda y Google Colab, este notebook escribe los kets y bras con LaTeX nativo expandido, sin depender de comandos abreviados externos. En particular:

$$
\left|x\right\rangle,
\qquad
\left\langle x\right|,
\qquad
\left|s\right\rangle\left\langle s\right|.
$$

Esta convención mantiene clara la diferencia entre un vector de estado, su dual y un proyector. Todas las fórmulas posteriores usan esta misma codificación.

## Objetivos de aprendizaje

Al finalizar, se espera que puedas:

1. Formular la búsqueda no estructurada como un problema de consulta con un predicado booleano.
2. Relacionar el número de qubits $n$ con el tamaño del espacio de búsqueda $N=2^n$.
3. Explicar por qué una búsqueda clásica sin estructura requiere escala lineal en el número de candidatos.
4. Construir el estado uniforme $\left|s\right\rangle=N^{-1/2}\sum_x\left|x\right\rangle$.
5. Interpretar el oráculo de fase $O_f\left|x\right\rangle=(-1)^{f(x)}\left|x\right\rangle$.
6. Explicar la difusión como reflexión alrededor de $\left|s\right\rangle$ y como inversión respecto a la media.
7. Describir una iteración de Grover como composición de dos reflexiones.
8. Justificar conceptualmente el número aproximado de iteraciones $r\approx \frac{\pi}{4}\sqrt{N/k}$.
9. Implementar ejemplos en Cirq y Qiskit.
10. Diagnosticar errores frecuentes en oráculos, difusión, número de iteraciones y orden de bits.

## 1. Problema de búsqueda no estructurada

Sea $N=2^n$ el número de candidatos. Cada candidato se etiqueta mediante una cadena binaria $x\in\{0,1\}^n$. La información sobre las soluciones se entrega mediante un predicado booleano

$$
f:\{0,1\}^n\to\{0,1\},
$$

donde $f(x)=1$ significa que $x$ es solución y $f(x)=0$ significa que no lo es. El conjunto marcado es

$$
M=\{x:f(x)=1\},\qquad k=|M|.
$$

El problema no consiste en calcular todos los valores de $f$. El objetivo es encontrar algún $x\in M$ usando el menor número posible de consultas al predicado. En ausencia de estructura, un algoritmo clásico que consulta candidatos individuales necesita escala $O(N/k)$ para obtener una solución con probabilidad constante. Grover reduce esa escala a $O(\sqrt{N/k})$ consultas en el modelo ideal.

In [ ]:
# Instalación condicional para JupyterLab, Anaconda o Google Colab.
# Si los paquetes ya existen, esta celda no modifica el entorno.
import importlib.util
import subprocess
import sys


def ensure_package(import_name, pip_name=None):
    """Instala un paquete solo si no puede importarse."""
    if importlib.util.find_spec(import_name) is None:
        pkg = pip_name or import_name
        print(f"Instalando {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    else:
        print(f"{import_name} disponible")

ensure_package("numpy")
ensure_package("matplotlib")
ensure_package("qiskit")
ensure_package("qiskit_aer", "qiskit-aer")
ensure_package("cirq")


In [ ]:
import math
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import cirq
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Para que los ejemplos sean reproducibles.
np.random.seed(7)


## 2. Estado uniforme

El algoritmo comienza sin preferir ningún candidato. Por ello prepara el estado uniforme

$$
\left|s\right\rangle=\frac{1}{\sqrt{N}}\sum_{x=0}^{N-1}\left|x\right\rangle.
$$

Si $N=2^n$, este estado se obtiene aplicando Hadamard a cada qubit inicializado en $\left|0\right\rangle$:

$$
H^{\otimes n}\left|0^n\right\rangle
=\left(\frac{\left|0\right\rangle+\left|1\right\rangle}{\sqrt{2}}\right)^{\otimes n}
=\frac{1}{\sqrt{2^n}}\sum_{x\in\{0,1\}^n}\left|x\right\rangle.
$$

La superposición uniforme no entrega todas las respuestas al medir. Si se mide inmediatamente, el resultado es casi uniforme. Su función es preparar caminos coherentes para que el oráculo y la difusión puedan modificar amplitudes.


### Desarrollo paso a paso: por qué $64$ candidatos requieren $6$ qubits

El registro de búsqueda de $n$ qubits tiene $2^n$ estados base. Para representar una lista de $64$ candidatos se busca $n$ tal que

$$
2^n=64.
$$

Calculamos las potencias relevantes:

$$
2^1=2,
\qquad
2^2=4,
\qquad
2^3=8,
\qquad
2^4=16,
\qquad
2^5=32,
\qquad
2^6=64.
$$

Por tanto, se requieren $6$ qubits. La inicialización coherente es

$$
H^{\otimes 6}\left|000000\right\rangle
=
\frac{1}{\sqrt{64}}
\sum_{x=0}^{63}\left|x\right\rangle.
$$

La conclusión conceptual es que los seis qubits no almacenan sesenta y cuatro respuestas clásicas accesibles por lectura directa. Preparan sesenta y cuatro caminos coherentes con igual amplitud inicial para que el oráculo pueda introducir una diferencia de fase y la difusión pueda convertir esa diferencia en un sesgo observable de probabilidad.


In [ ]:
def uniform_state(n):
    """Devuelve el vector de estado uniforme para n qubits."""
    N = 2 ** n
    return np.ones(N, dtype=complex) / np.sqrt(N)

n = 6
state = uniform_state(n)
print("Número de qubits:", n)
print("Número de candidatos N:", 2**n)
print("Norma del estado uniforme:", np.linalg.norm(state))
print("Probabilidad inicial de un candidato específico:", abs(state[0])**2)


## 3. Oráculo de fase

El oráculo de Grover marca soluciones mediante una fase:

$$
O_f\left|x\right\rangle=(-1)^{f(x)}\left|x\right\rangle.
$$

Si $x$ es solución, $f(x)=1$ y el estado base cambia de signo. Si no lo es, $f(x)=0$ y queda igual. Este cambio de signo no aumenta todavía la probabilidad de medir la solución, porque $|a|^2=|-a|^2$. Su papel es crear un contraste de fase que la difusión puede convertir en amplitud.

En muchas implementaciones, este oráculo de fase se obtiene desde un oráculo reversible con auxiliar:

$$
U_f\left|x\right\rangle\left|y\right\rangle=\left|x\right\rangle\left|y\oplus f(x)\right\rangle.
$$

Si el auxiliar está en $\left|-\right\rangle$, entonces

$$
U_f\left|x\right\rangle\left|-\right\rangle=(-1)^{f(x)}\left|x\right\rangle\left|-\right\rangle.
$$

In [ ]:
def phase_oracle_statevector(state, marked_indices):
    """Aplica un oráculo de fase a un vector de estado.

    Los índices marcados reciben un factor -1.
    """
    new_state = state.copy()
    for idx in marked_indices:
        new_state[idx] *= -1
    return new_state

n = 3
marked = [int("101", 2)]
psi = uniform_state(n)
psi_after_oracle = phase_oracle_statevector(psi, marked)

print("Estado marcado:", format(marked[0], "03b"))
print("Amplitud antes del oráculo:", psi[marked[0]])
print("Amplitud después del oráculo:", psi_after_oracle[marked[0]])
print("Probabilidad antes:", abs(psi[marked[0]])**2)
print("Probabilidad después:", abs(psi_after_oracle[marked[0]])**2)


## 4. Difusión como inversión respecto a la media

Sea

$$
\left|\psi\right\rangle=\sum_x a_x\left|x\right\rangle.
$$

La media de las amplitudes es

$$
\bar a=\frac{1}{N}\sum_x a_x.
$$

El operador de difusión

$$
D=2\left|s\right\rangle\left\langle s\right|-I
$$

actúa sobre las amplitudes como

$$
a_x\longmapsto 2\bar a-a_x.
$$

Esta es una inversión respecto a la media. Después del oráculo, las amplitudes de los estados marcados quedan por debajo de la media debido al cambio de signo; al reflejarlas respecto a la media, pasan a ser mayores que las demás. Así se convierte una diferencia de fase en una diferencia de probabilidad.

In [ ]:
def diffusion_statevector(state):
    """Aplica inversión respecto a la media a un vector de amplitudes."""
    mean = np.mean(state)
    return 2 * mean - state

n = 3
marked = [int("101", 2)]
psi = uniform_state(n)
psi_oracle = phase_oracle_statevector(psi, marked)
psi_diffused = diffusion_statevector(psi_oracle)

labels = [format(i, "03b") for i in range(2**n)]
print("Amplitudes después del oráculo:")
for label, amp in zip(labels, psi_oracle):
    print(label, np.round(amp.real, 4))

print("\nAmplitudes después de la difusión:")
for label, amp in zip(labels, psi_diffused):
    print(label, np.round(amp.real, 4))



### Desarrollo paso a paso: una iteración con $N=8$ y una solución marcada

Supongamos un registro de tres qubits, por lo que $N=2^3=8$. Si hay una única solución marcada, la amplitud inicial de cada estado base en la superposición uniforme es

$$
a=\frac{1}{\sqrt{8}}.
$$

Después del oráculo, la solución marcada tiene amplitud $-a$, mientras que los siete estados no marcados conservan amplitud $a$. La media de las amplitudes queda

$$
\bar a
=
\frac{7a+(-a)}{8}
=
\frac{6a}{8}
=
\frac{3a}{4}.
$$

La difusión aplica la regla

$$
a_x \longmapsto 2\bar a-a_x.
$$

Para el estado marcado:

$$
-a
\longmapsto
2\left(\frac{3a}{4}\right)-(-a)
=
\frac{3a}{2}+a
=
\frac{5a}{2}.
$$

Para cada estado no marcado:

$$
a
\longmapsto
2\left(\frac{3a}{4}\right)-a
=
\frac{3a}{2}-a
=
\frac{a}{2}.
$$

La probabilidad de la solución marcada después de esta iteración es

$$
\left|\frac{5a}{2}\right|^2
=
\frac{25}{4}|a|^2
=
\frac{25}{4}\cdot\frac{1}{8}
=
\frac{25}{32}.
$$

El cálculo muestra el propósito de la difusión: convertir un signo introducido por el oráculo en una amplificación real de probabilidad. El oráculo solo crea contraste de fase; la difusión transforma ese contraste en una distribución medible favorable a la solución.


In [ ]:
# Visualización sencilla de amplitudes reales antes y después de la difusión.
plt.figure(figsize=(8, 4))
plt.bar(labels, psi_diffused.real)
plt.xlabel("Estado base")
plt.ylabel("Amplitud real")
plt.title("Después de una iteración parcial: oráculo + difusión")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5. Iteración de Grover y número de repeticiones

Una iteración de Grover es la composición

$$
G=DO_f.
$$

El oráculo invierte la fase del subespacio marcado y la difusión refleja alrededor del estado uniforme. La composición de dos reflexiones es una rotación. Si hay $k$ soluciones en $N$ candidatos, se define

$$
\sin^2\theta=\frac{k}{N}.
$$

Después de $r$ iteraciones, la probabilidad total de medir una solución idealmente es

$$
P_r=\sin^2((2r+1)\theta).
$$

El máximo ocurre cuando $(2r+1)\theta$ queda cerca de $\pi/2$. Para $k\ll N$ se usa la regla aproximada

$$
r\approx \frac{\pi}{4}\sqrt{\frac{N}{k}}.
$$

Esta regla explica por qué hay sobrerotación: si se sigue iterando después del máximo, la probabilidad de éxito puede disminuir.

In [ ]:
def recommended_iterations(N, k):
    """Número entero recomendado de iteraciones para Grover básico."""
    if k <= 0 or k > N:
        raise ValueError("Se requiere 1 <= k <= N")
    return max(1, int(round((math.pi / 4) * math.sqrt(N / k))))

for k in [1, 2, 4, 8]:
    print(f"N=64, k={k}: iteraciones recomendadas = {recommended_iterations(64, k)}")


In [ ]:
def grover_statevector(n, marked_bitstrings, iterations=None):
    """Simulación vectorial básica de Grover.

    Esta función no usa Qiskit ni Cirq; sirve para observar la dinámica matemática.
    """
    N = 2 ** n
    marked_indices = [int(b, 2) for b in marked_bitstrings]
    if iterations is None:
        iterations = recommended_iterations(N, len(marked_indices))
    state = uniform_state(n)
    history = [state.copy()]
    for _ in range(iterations):
        state = phase_oracle_statevector(state, marked_indices)
        state = diffusion_statevector(state)
        history.append(state.copy())
    return state, history

final_state, history = grover_statevector(3, ["101"], iterations=2)
probs = np.abs(final_state)**2
for i, p in enumerate(probs):
    print(format(i, "03b"), round(float(p), 4))



## 6. Ejemplo desarrollado: marcar $\left|101\right\rangle$

**Problema.** Queremos encontrar la cadena $101$ dentro de un registro de tres qubits. El oráculo debe implementar

$$
O\left|101\right\rangle=-\left|101\right\rangle,
\qquad
O\left|x\right\rangle=\left|x\right\rangle\quad (x\neq 101).
$$

**Idea cuántica.** El oráculo no revela la solución como un dato clásico. Cambia el signo de la amplitud asociada con la solución. Esa diferencia de fase será útil solo después de aplicar difusión.

**Desarrollo paso a paso del marcado.** La fase multi-controlada estándar se activa naturalmente sobre el patrón $111$. Como queremos marcar $101$, convertimos temporalmente el cero de la segunda posición en un uno:

$$
\left|101\right\rangle
\xrightarrow{X\text{ en el segundo qubit}}
\left|111\right\rangle.
$$

Ahora la fase multi-controlada sí se activa:

$$
\left|111\right\rangle\longmapsto -\left|111\right\rangle.
$$

Finalmente se deshace la traducción temporal:

$$
-\left|111\right\rangle
\xrightarrow{X\text{ en el segundo qubit}}
-\left|101\right\rangle.
$$

**Justificación.** Los $X$ antes y después no son parte del predicado final; son una herramienta para implementar controles negativos. El paso de deshacer es indispensable porque un oráculo de búsqueda debe marcar la etiqueta sin cambiarla.

**Interpretación.** El resultado final no significa que el circuito haya leído una lista de ocho valores. Significa que la solución definida por el predicado quedó distinguida por fase y lista para ser amplificada por la difusión.



### Variación desarrollada: marcar $\left|101\right\rangle$ y $\left|111\right\rangle$

Ahora el conjunto marcado es

$$
M=\{101,111\}.
$$

Las dos cadenas comparten el primer y el tercer bit igual a $1$; el segundo bit puede ser $0$ o $1$. Por tanto, el predicado puede escribirse como

$$
f(x_1x_2x_3)=1
\quad\Longleftrightarrow\quad
x_1=1\ \text{y}\ x_3=1.
$$

El segundo qubit no debe usarse como control, porque exigirlo eliminaría una de las dos soluciones. La acción deseada es

$$
\left|101\right\rangle\mapsto -\left|101\right\rangle,
\qquad
\left|111\right\rangle\mapsto -\left|111\right\rangle,
$$

mientras que los demás estados deben quedar sin cambio de signo. Esta variación ilustra una diferencia central entre conocer una solución y conocer un predicado: el circuito implementa una condición lógica sobre bits, no necesariamente una lista explícita de estados escritos uno por uno.


## 7. Implementación en Cirq

In [ ]:
def mcz_cirq(qubits):
    """Compuerta Z multi-controlada sobre el último qubit."""
    if len(qubits) == 1:
        return cirq.Z(qubits[0])
    return cirq.Z(qubits[-1]).controlled_by(*qubits[:-1])


def mark_state_cirq_ops(qubits, bitstring):
    """Operaciones de Cirq que marcan una cadena con fase negativa."""
    ops = []
    for q, bit in zip(qubits, bitstring):
        if bit == "0":
            ops.append(cirq.X(q))
    ops.append(mcz_cirq(qubits))
    for q, bit in zip(qubits, bitstring):
        if bit == "0":
            ops.append(cirq.X(q))
    return ops


def diffusion_cirq_ops(qubits):
    """Difusión de Grover sobre el registro de búsqueda."""
    ops = []
    ops.extend(cirq.H.on_each(*qubits))
    ops.extend(cirq.X.on_each(*qubits))
    ops.append(mcz_cirq(qubits))
    ops.extend(cirq.X.on_each(*qubits))
    ops.extend(cirq.H.on_each(*qubits))
    return ops


def grover_cirq_circuit(marked_bitstrings, n, iterations=None):
    """Construye un circuito de Grover en Cirq."""
    qubits = cirq.LineQubit.range(n)
    N = 2 ** n
    if iterations is None:
        iterations = recommended_iterations(N, len(marked_bitstrings))
    circuit = cirq.Circuit()
    circuit.append(cirq.H.on_each(*qubits))
    for _ in range(iterations):
        for bitstring in marked_bitstrings:
            circuit.append(mark_state_cirq_ops(qubits, bitstring))
        circuit.append(diffusion_cirq_ops(qubits))
    circuit.append(cirq.measure(*qubits, key="x"))
    return circuit


def run_cirq_counts(circuit, n, repetitions=1000):
    """Ejecuta un circuito Cirq y devuelve conteos como cadenas binarias."""
    simulator = cirq.Simulator(seed=7)
    result = simulator.run(circuit, repetitions=repetitions)
    rows = result.measurements["x"]
    strings = ["".join(str(int(bit)) for bit in row) for row in rows]
    return Counter(strings)

circuit_cirq = grover_cirq_circuit(["101"], n=3, iterations=2)
print(circuit_cirq)
counts_cirq = run_cirq_counts(circuit_cirq, n=3, repetitions=1000)
counts_cirq


In [ ]:
# Caso con dos soluciones: 101 y 111.
circuit_cirq_two = grover_cirq_circuit(["101", "111"], n=3, iterations=1)
counts_cirq_two = run_cirq_counts(circuit_cirq_two, n=3, repetitions=1000)
counts_cirq_two


## 8. Implementación en Qiskit

La implementación en Qiskit sigue la misma lógica. Se define una fase multi-controlada, un marcador para cadenas concretas, el operador de difusión y el circuito completo. La medición se realiza con un orden explícito para que la cadena impresa coincida con el orden conceptual usado al escribir las soluciones.

In [ ]:
def apply_mcz_qiskit(qc, qubits):
    """Aplica una Z multi-controlada sobre el último qubit de la lista."""
    if len(qubits) == 1:
        qc.z(qubits[0])
    else:
        qc.h(qubits[-1])
        qc.mcx(qubits[:-1], qubits[-1])
        qc.h(qubits[-1])


def mark_state_qiskit(qc, bitstring):
    """Marca una cadena con fase negativa en Qiskit.

    Convención: bitstring[0] corresponde al qubit 0.
    """
    n = len(bitstring)
    qubits = list(range(n))
    for i, bit in enumerate(bitstring):
        if bit == "0":
            qc.x(i)
    apply_mcz_qiskit(qc, qubits)
    for i, bit in enumerate(bitstring):
        if bit == "0":
            qc.x(i)


def diffusion_qiskit(qc, n):
    """Difusión de Grover sobre n qubits de búsqueda."""
    qc.h(range(n))
    qc.x(range(n))
    apply_mcz_qiskit(qc, list(range(n)))
    qc.x(range(n))
    qc.h(range(n))


def grover_qiskit_circuit(marked_bitstrings, n, iterations=None):
    """Construye el circuito completo de Grover en Qiskit."""
    N = 2 ** n
    if iterations is None:
        iterations = recommended_iterations(N, len(marked_bitstrings))
    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    for _ in range(iterations):
        for bitstring in marked_bitstrings:
            mark_state_qiskit(qc, bitstring)
        diffusion_qiskit(qc, n)
    for i in range(n):
        qc.measure(i, n - 1 - i)
    return qc


def run_qiskit_counts(qc, shots=1000):
    """Ejecuta un circuito Qiskit con AerSimulator y devuelve conteos."""
    simulator = AerSimulator(seed_simulator=7)
    tqc = transpile(qc, simulator)
    result = simulator.run(tqc, shots=shots).result()
    return result.get_counts()

qc = grover_qiskit_circuit(["101"], n=3, iterations=2)
print(qc)
counts_qiskit = run_qiskit_counts(qc, shots=1000)
counts_qiskit


In [ ]:
qc_two = grover_qiskit_circuit(["101", "111"], n=3, iterations=1)
counts_qiskit_two = run_qiskit_counts(qc_two, shots=1000)
counts_qiskit_two



## 9. Ejercicio guiado 1: controles negativos

**Enunciado.** En un registro de tres qubits se desea marcar $010$. ¿Qué qubits requieren controles negativos?

**Desarrollo paso a paso.** La fase multi-controlada estándar reconoce el patrón $111$. Para que el patrón $010$ active esa fase, se deben convertir temporalmente sus ceros en unos. La cadena objetivo es

$$
010.
$$

Los ceros están en la primera y tercera posiciones, por lo que se aplican $X$ en esos qubits:

$$
\left|010\right\rangle
\xrightarrow{X_1X_3}
\left|111\right\rangle.
$$

Después se aplica la fase multi-controlada:

$$
\left|111\right\rangle\longmapsto -\left|111\right\rangle.
$$

Luego se deshacen los $X$:

$$
-\left|111\right\rangle
\xrightarrow{X_1X_3}
-\left|010\right\rangle.
$$

**Justificación.** La fase multi-controlada no sabe reconocer ceros. Los $X$ convierten temporalmente controles negativos en controles positivos. La inversión final restaura la etiqueta original.

**Interpretación.** El oráculo debe marcar exactamente el predicado deseado. Si se omite un control negativo, Grover amplificará el estado que realmente fue marcado, no el estado que se pretendía marcar.


In [ ]:
qc_010 = grover_qiskit_circuit(["010"], n=3, iterations=2)
run_qiskit_counts(qc_010, shots=1000)



## 10. Ejercicio guiado 2: estimar iteraciones

**Enunciado.** Para $N=64$ y $k=4$, estima el número de iteraciones de Grover.

**Desarrollo paso a paso.** La regla aproximada es

$$
r\approx \frac{\pi}{4}\sqrt{\frac{N}{k}}.
$$

Sustituimos $N=64$ y $k=4$:

$$
r\approx \frac{\pi}{4}\sqrt{\frac{64}{4}}.
$$

Primero se simplifica el cociente:

$$
\frac{64}{4}=16.
$$

Después se calcula la raíz:

$$
\sqrt{16}=4.
$$

Por tanto,

$$
r\approx \frac{\pi}{4}\cdot 4=\pi\approx 3.14.
$$

Como el circuito solo puede aplicar un número entero de iteraciones, se elige un entero cercano, típicamente $3$.

**Interpretación.** Con cuatro soluciones, el subespacio marcado ocupa una fracción mayor del espacio que cuando $k=1$. Por eso se necesitan menos iteraciones: el estado inicial ya tiene una componente marcada más grande y requiere una rotación menor para acercarse al máximo de probabilidad.


In [ ]:
print("Iteraciones recomendadas para N=64, k=4:", recommended_iterations(64, 4))



## 11. Ejercicio guiado 3: sobrerrotación

**Enunciado.** Para $N=8$ y una solución marcada, compara la probabilidad de éxito al usar una, dos, tres y cuatro iteraciones.

**Desarrollo matemático.** Con $N=8$ y $k=1$,

$$
\sin^2\theta=\frac{k}{N}=\frac{1}{8},
\qquad
\theta=\arcsin\left(\frac{1}{\sqrt{8}}\right).
$$

Después de $r$ iteraciones, la probabilidad total de medir una solución es

$$
P_r=\sin^2((2r+1)\theta).
$$

Los valores aproximados son:

$$
P_0=\frac{1}{8},
\qquad
P_1\approx 0.781,
\qquad
P_2\approx 0.945,
\qquad
P_3\approx 0.330.
$$

**Justificación.** La composición oráculo-difusión es una rotación en el plano marcado/no marcado. Al principio la rotación acerca el estado al subespacio marcado. Después de pasar el punto óptimo, seguir rotando lo aleja.

**Interpretación.** Si una simulación empeora al aumentar iteraciones, no necesariamente hay un error de programación. Puede estar ocurriendo sobrerrotación. Grover no acumula probabilidad de forma irreversible; redistribuye amplitudes mediante operaciones unitarias.


In [ ]:
marked = ["101"]
for r in [1, 2, 3, 4]:
    final_state, _ = grover_statevector(3, marked, iterations=r)
    prob_success = abs(final_state[int(marked[0], 2)])**2
    print(f"iteraciones={r}, probabilidad de éxito={prob_success:.4f}")



## 12. Ejercicios propuestos con orientación matemática

1. **Construye un oráculo que marque $011$.** Identifica primero los ceros de la cadena objetivo. En $011$ solo la primera posición es cero, por lo que se aplica $X$ en el primer qubit, después la fase multi-controlada y finalmente se deshace ese $X$.

2. **Para $N=128$ y $k=2$, estima el número de iteraciones recomendado.** Usa

   $$
   r\approx \frac{\pi}{4}\sqrt{\frac{N}{k}}
   =\frac{\pi}{4}\sqrt{\frac{128}{2}}
   =\frac{\pi}{4}\sqrt{64}
   =2\pi\approx 6.28.
   $$

   Un entero cercano es $6$.

3. **Simula en Qiskit un caso con soluciones $001$ y $110$.** La interpretación correcta del histograma no es que el circuito haya probado todas las entradas una por una. El histograma debe concentrarse en las cadenas que el oráculo marcó porque la difusión amplifica el subespacio marcado.

4. **Modifica deliberadamente un oráculo para marcar un estado equivocado.** Si el histograma amplifica otro estado, la conclusión no es que Grover falló. La conclusión es que el predicado implementado no coincide con el predicado deseado.

5. **Explica por qué medir después del oráculo destruye la ventaja.** El oráculo cambia $a_x$ por $(-1)^{f(x)}a_x$, pero la probabilidad sigue siendo

   $$
   |(-1)^{f(x)}a_x|^2=|a_x|^2.
   $$

   La información todavía está en fase. Medir en ese punto elimina la coherencia antes de que la difusión convierta esa fase en probabilidad.


## 13. Resumen final

Grover resuelve búsqueda no estructurada por predicado. El algoritmo no inspecciona todos los candidatos como datos clásicos; prepara una superposición uniforme, marca soluciones por fase, aplica difusión para convertir fase en amplitud y repite ese proceso un número cuidadosamente elegido de veces.

Las fórmulas esenciales son

$$
\left|s\right\rangle=\frac{1}{\sqrt{N}}\sum_x\left|x\right\rangle,
\qquad
O_f\left|x\right\rangle=(-1)^{f(x)}\left|x\right\rangle,
\qquad
D=2\left|s\right\rangle\left\langle s\right|-I,
\qquad
r\approx\frac{\pi}{4}\sqrt{\frac{N}{k}}.
$$

La interpretación profunda es geométrica: el oráculo y la difusión son reflexiones, y su composición produce una rotación hacia el subespacio marcado. La ventaja frente a búsqueda clásica no estructurada es cuadrática en número de consultas.